# 02 · Estrategias

## Paso 3 · Estrategias base y señales sin look-ahead

**Regla central:** la señal calculada con el cierre del día *t* se ejecuta en *t+1*. Cada estrategia devuelve, por ETF, cuánto invertir (entre 0 y 1) usando solo información disponible hasta el cierre de cada día.

| Estrategia | Idea | Parámetros fijos (literatura, no optimizados) |
|---|---|---|
| Buy & hold | Benchmark: siempre invertido | — |
| Cruce de medias | Invertido si la media de 50 d está sobre la de 200 d | 50 / 200 |
| Momentum 12-1 | Invertido si el retorno de 12 meses sin el último mes es positivo | 252 / 21 d |
| Reversión a la media | Invertido tras caer más de 1 desv. est. bajo la media de 20 d | 20 d, z = −1 |
| Volatilidad objetivo | Siempre invertido, con exposición = 10 % / vol. de 21 d (máx. 1) | 10 %, 21 d |

In [1]:
import sys; sys.path.append("..")
import pandas as pd
from src.data import download_prices
from src.signals import STRATEGIES

px = download_prices(cache="../data/raw/prices.parquet")
print(px.shape, px.index.min().date(), px.index.max().date())

(5282, 5) 2005-01-03 2025-12-30


### 3.2 Prueba anti look-ahead
Una señal honesta calculada con datos recortados debe coincidir con la calculada con todos los datos en las fechas comunes. Si una estrategia usara precios futuros, cambiar el final de la serie alteraría el pasado y la prueba fallaría.

In [2]:
def check_no_lookahead(signal_fn, prices, cut=2000):
    full = signal_fn(prices)
    trunc = signal_fn(prices.iloc[:cut])
    pd.testing.assert_frame_equal(full.iloc[:cut], trunc)

for name, fn in STRATEGIES.items():
    for cut in [300, 2000, 4000]:            # varios cortes, no solo uno
        check_no_lookahead(fn, px, cut)
    print(name, "ok")

buy_and_hold ok
sma_cross ok
momentum ok
mean_reversion ok
vol_target ok


### Control de cordura de las señales
Rango de valores y exposición media tras el calentamiento. Ojo: `sma_cross`, `momentum` y `mean_reversion` devuelven **0 (efectivo)** mientras sus indicadores aún no existen (una comparación con NaN da `False`), no NaN. Por eso no basta con mirar dónde empieza la señal: hay que contar la historia que necesita cada indicador.

In [3]:
import numpy as np
# Primer día en que el indicador subyacente de cada estrategia existe (medido, no supuesto)
first_valid = {
    "buy_and_hold": px.index[0],
    "sma_cross": px.rolling(200).mean().first_valid_index(),
    "momentum": (px.shift(21) / px.shift(252)).first_valid_index(),
    "mean_reversion": px.rolling(20).std().first_valid_index(),
    "vol_target": px.pct_change().rolling(21).std().first_valid_index(),
}
start = px.index[260]
rows = []
for name, fn in STRATEGIES.items():
    s = fn(px)
    rows.append({"estrategia": name, "min": s.min().min(), "max": s.max().max(),
                 "dias_de_historia": px.index.get_loc(first_valid[name]),
                 "primera_senal_real": first_valid[name].date(),
                 "exposicion_media": s.loc[start:].mean().mean()})
pd.DataFrame(rows).set_index("estrategia").round(3)

,min,max,dias_de_historia,primera_senal_real,exposicion_media
estrategia,,,,,
buy_and_hold,1.000,1.0,0,2005-01-03,1.000
sma_cross,0.000,1.0,199,2005-10-17,0.693
momentum,0.000,1.0,252,2006-01-03,0.718
mean_reversion,0.000,1.0,19,2005-01-31,0.207
vol_target,0.061,1.0,21,2005-02-02,0.669


In [4]:
assert max(first_valid.values()) < start
print("Fecha común de inicio de la evaluación (tras 260 días de calentamiento):", start.date())

Fecha común de inicio de la evaluación (tras 260 días de calentamiento): 2006-01-13


### 3.3 Decisiones de diseño (fijadas antes de ver resultados)

- **Parámetros fijos de la literatura** (tabla de arriba), elegidos antes de correr ningún backtest. Cualquier variante que se pruebe después cuenta como un intento más y se anota en `EXPERIMENTS.md` (lo necesita el Deflated Sharpe del Paso 7).
- **Cartera:** cada ETF recibe 1/5 del capital y su señal decide qué fracción de ese quinto se invierte. Lo no invertido queda en efectivo, que rinde **0 %** (supuesto simplificador; es una limitación, sobre todo frente a tasas reales del 4-5 % en 2023-2025).
- **Calentamiento:** la señal con más historia es el momentum (252 días). Todas las estrategias se evalúan desde la misma fecha, el día 260 (**2006-01-13**), para que la comparación sea justa. Sin esto, el cruce de medias y el momentum aparecerían "en efectivo" durante 2005 solo por falta de datos.
- **Solo largo, sin apalancamiento:** posiciones entre 0 y 1 (comprobado arriba: mínimo ≥ 0 y máximo = 1). Es más realista para un inversionista individual y evita costos de préstamo.
- **Exposición media muy distinta:** buy & hold 100 %, momentum ~72 %, cruce de medias ~69 %, volatilidad objetivo ~67 % y reversión a la media solo ~21 %. Más adelante habrá que separar la ventaja de la señal del simple hecho de estar más o menos invertido (placebo de timing, Paso 7).
